## **Spectroscopic IROS Reconstruction**

In [1]:
from pathlib import Path

from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.io import simulation_files
import darksun as ds

ds.show.set_figures_darkbkg()

In [2]:
# BASE_PATH: str = "/mnt/dbb8f47e-da06-47bf-8ef5-038092af70f7/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Data"
# SIM_PATH: str = f"{BASE_PATH}/Simulations"
# OUT_PATH: str = f"{BASE_PATH}/Outputs"

BASE_PATH: str = "/mnt/d/PhD_AASS/Coding/Images_fits"
SIM_PATH = OUT_PATH = BASE_PATH

In [3]:
# MASK_FITS: str = "mask_NTHT_20260129_CORRECTED.fits"
MASK_FITS: str = "mask_NTHT_20250725.fits"
UPS_X: int = 2
UPS_Y: int = 1

wfm: CodedMaskCamera = codedmask(f"{SIM_PATH}/{MASK_FITS}", UPS_X, UPS_Y)

In [4]:
# SKYFIELD: str = "IROSDummy"
# DATA_FITS: str = "iros_benchmark_2-50keV_mask_050_1040x17_1ks"
SKYFIELD: str = "GalacticCentre"
DATA_FITS: str = "galctr_rxte-sax_mask_050_1040x17_2-50keV_1ks"

ID_CAMERA_A: str = "cam1a"
DATASET: str = "reconstructed"

filepaths: dict[str, dict[str, Path]] = simulation_files(f"{SIM_PATH}/{SKYFIELD}/{DATA_FITS}")

In [5]:
from typing import Any

import numpy as np
from numpy.typing import NDArray
from astropy.io.fits import FITS_rec
from tqdm import tqdm

from bloodmoon.types import CoordEquatorial
from darksun.data import Log
from darksun.benchmarking import source_catalogue_data
from darksun.data import DataLoader, CatalogueLoader

from IROSrec.iros.optim import iros_singleCAM
from IROSrec.iros.procedure import run_IROS, get_sources_database

In [6]:
# - data gathering from specific direction
#   NOTE: Filtering the whole photons list by specific incoming direction associated with a source
#         is just a way to reduce the computational cost of data handling operations.
#         Since we are dealing with a WM sim, we know the actual photons directions. In a real
#         obs this info is lost in the encoding phase (hence, we can only filter for energy).
#   NOTE: If the spectroscopic analysis is performed AFTER a first IROS reconstruction, the sources
#         coords in the camera local-frame are known, and can be utilised in the sky image to select
#         subsections of the skymap for the spectroscopic analysis (e.g., as an external trigger).

def get_source_coords(sourceID: str, catalogue: CatalogueLoader) -> CoordEquatorial:
    """Extracts the source RA/Dec coords from catalogue."""
    data = source_catalogue_data(sourceID, catalogue.DLdata)
    return CoordEquatorial(data['RA'], data['DEC'])

def select_target_srcs(
    photons: FITS_rec,
    srcIDs: str | tuple[str, ...],
    catalogue: CatalogueLoader,
) -> FITS_rec:
    """Filters the input photon list for the given sources equatorial coords."""
    srcIDs_ = (srcIDs,) if isinstance(srcIDs, str) else srcIDs
    coords = tuple(get_source_coords(src, catalogue) for src in srcIDs_)
    phs = ds.select_source_photons(coords, photons, False)
    return phs

In [7]:
# - energy bands scheduler and filtering

type EnergyRange = tuple[float, float]
type EnergySchedule = dict[int, EnergyRange]

def eband_schedule(emin: float, emax: float, ebin: float) -> EnergySchedule:
    """Defines a schedule for the energy bands to analyse."""
    if not (emax > emin) or (emax - emin < ebin):
        raise ValueError('Invalid energy band boundary values.')
     
    nruns = int((emax - emin) / ebin)
    schedule = {run: (float(emin), float(emax - run * ebin)) for run in range(nruns)}
    return schedule

def select_eband_phs(photons: FITS_rec, eband: EnergyRange) -> FITS_rec:
    """Filters the input photon list in the given energy band."""
    E_min, E_max = eband
    return ds.filter_data(photons, E_min=E_min, E_max=E_max, coords=None)

In [ ]:
from bloodmoon.mask import count, variance

# - define operations for single energy band IROS run

def perform_IROS(
    camera: CodedMaskCamera,
    detector: NDArray,
    max_iterations: int,
    camID: str | None = None,
    **iros_kwargs: Any,
) -> Log:
    """Runs the IROS procedure and stores optimised sources params."""
    loop = iros_singleCAM(camera, detector, max_iterations, **iros_kwargs)
    log, _ = run_IROS(camera, loop, camID)
    return log


def run_IROS_spectr_rec(
    camera: CodedMaskCamera,
    sdl: DataLoader,
    srcIDs: str | tuple[str, ...],
    schedule: EnergySchedule,
    catalogue: CatalogueLoader,
    vignetting: bool = True,
    screening: bool = True,
    **iros_kwargs: Any,
) -> dict[EnergyRange, Log]:
    """
    Performs the spectroscopic IROS reconstruction in the specified energy bands.

    NOTE:
        * The BKG is filtered out (if CXB phs are flagged, they can be kept
          in the data by adding the flag ID to the `srcIDs` sequence).
    """
    out: dict[EnergyRange, Log] = {}

    # filter for sources
    photons = select_target_srcs(sdl.DLdata, srcIDs, catalogue)

    # run eband reconstr
    _det, _ = count(camera, sdl.DLdata)
    varmap = variance(camera, _det)
    max_iterations = 2 * len(srcIDs)
    rec_loop = tqdm(list(schedule.items()))

    for runID, eband in rec_loop:
        rec_loop.set_description(f'Run {runID} | IROS reconstr - {eband} keV')
        phs = select_eband_phs(photons, eband)
        eband_detector, _ = count(camera, phs)
        log = perform_IROS(camera, eband_detector, max_iterations, varmap=varmap, **iros_kwargs)
        log = get_sources_database(camera, sdl, catalogue, log, vignetting, screening)
        out[eband] = log
    
    return out

In [ ]:
sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET])
catA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])
schedule: EnergySchedule = eband_schedule(2.0, 10.0, 2.0)

print(schedule)

{0: (2.0, 10.0), 1: (2.0, 8.0), 2: (2.0, 6.0), 3: (2.0, 4.0)}


In [13]:
srcIDs = (
    'scox1', 'gx5-1',
)

out_logs = run_IROS_spectr_rec(
    camera=wfm,
    sdl=sdlA,
    srcIDs=srcIDs,
    schedule=schedule,
    catalogue=catA,
) 

Run 0 | IROS reconstr - (2.0, 10.0) keV:   0%|          | 0/4 [00:00<?, ?it/s]

# Looping around the FOV...



## Optimisation Results:
  * p[0]:
      - START.: 43.8750000
      - OPTIM.: 43.9136430 +/- 0.0000368
      - GAIN.: 0.088 %
  * p[1]:
      - START.: 78.0000000
      - OPTIM.: 78.0552428 +/- 0.0021413
      - GAIN.: 0.071 %
  * p[2]:
      - START.: 1004378.2016448
      - OPTIM.: 955853.6173301 +/- 249.7761032
      - GAIN.: -4.831 %

## Fit Report:
  - func calls (also # of iters): 7
  - procedure msg: `ftol` termination condition is satisfied.
  - success status 2: Convergence in parameter values




## Optimisation Results:
  * p[0]:
      - START.: 13.5000000
      - OPTIM.: 13.5288537 +/- 0.0003577
      - GAIN.: 0.214 %
  * p[1]:
      - START.: -12.5000000
      - OPTIM.: -12.6066829 +/- 0.0190598
      - GAIN.: -0.853 %
  * p[2]:
      - START.: 108082.1077493
      - OPTIM.: 105476.2096464 +/- 255.9536266
      - GAIN.: -2.411 %

## Fit Report:
  - func calls (also # of iters): 7
  - procedure msg: Both `ftol` and `xtol` termination conditions are satisfied.
  - success status 4: Convergence in orthogonality




## Optimisation Results:
  * p[0]:
      - START.: 29.8750000
      - OPTIM.: 29.8533551 +/- 0.0041050
      - GAIN.: -0.072 %
  * p[1]:
      - START.: -15.0000000
      - OPTIM.: -14.5000000 +/- 0.2319074
      - GAIN.: 3.333 %
  * p[2]:
      - START.: 7930.8159695
      - OPTIM.: 7206.2896724 +/- 209.2588974
      - GAIN.: -9.136 %

## Fit Report:
  - func calls (also # of iters): 6
  - procedure msg: `ftol` termination condition is satisfied.
  - success status 2: Convergence in parameter values



...


## Optimisation Results:
  * p[0]:
      - START.: 43.8750000
      - OPTIM.: 43.9504378 +/- 0.0041412
      - GAIN.: 0.172 %
  * p[1]:
      - START.: 64.5000000
      - OPTIM.: 64.0473123 +/- 0.2517702
      - GAIN.: -0.702 %
  * p[2]:
      - START.: 8628.0603441
      - OPTIM.: 6967.1704949 +/- 209.9512244
      - GAIN.: -19.250 %

## Fit Report:
  - func calls (also # of iters): 21
  - procedure msg: `xtol` termination condition is satisfied.
  - success status 3: Convergence in both chi-square and parameter values



IROS sky-reconstruction: 4it [00:37,  9.34s/it]


## Optimisation Results:
  * p[0]:
      - START.: 43.8750000
      - OPTIM.: 43.8459225 +/- 0.0048547
      - GAIN.: -0.066 %
  * p[1]:
      - START.: 91.5000000
      - OPTIM.: 91.1442129 +/- 0.2958986
      - GAIN.: -0.389 %
  * p[2]:
      - START.: 7309.5504610
      - OPTIM.: 5902.4619972 +/- 208.0702635
      - GAIN.: -19.250 %

## Fit Report:
  - func calls (also # of iters): 7
  - procedure msg: `ftol` termination condition is satisfied.
  - success status 2: Convergence in parameter values




Run 3 | IROS reconstr - (2.0, 4.0) keV: 100%|██████████| 4/4 [01:35<00:00, 23.98s/it]

# Comparing with Catalogue...
# Successful comparison!


In [14]:
out_logs

{0: ((2.0, 10.0), <darksun.data.Log at 0x7f68e5c414f0>),
 1: ((2.0, 8.0), <darksun.data.Log at 0x7f68e5c7eb10>),
 2: ((2.0, 6.0), <darksun.data.Log at 0x7f68e5c80c20>),
 3: ((2.0, 4.0), <darksun.data.Log at 0x7f68e5cc4e60>)}

In [ ]:
for runID, (eband, log) in enumerate(out_logs.items()):
    print(f'Run {runID}, eband {eband} keV  |  Sources: {log.log['ID']} (brightest: {log.log['ID_brightest']})\n')

Run 0, eband (2.0, 10.0) keV  |  Sources: ['scox1', 'gx5-1', 'gx9+1'] (brightest: ['scox1', 'gx5-1', 'gx9+1'])

Run 1, eband (2.0, 8.0) keV  |  Sources: ['scox1', 'gx5-1'] (brightest: ['scox1', 'gx5-1'])

Run 2, eband (2.0, 6.0) keV  |  Sources: ['scox1', 'gx5-1'] (brightest: ['scox1', 'gx5-1'])

Run 3, eband (2.0, 4.0) keV  |  Sources: ['scox1', 'gx5-1', 'lemx-S1', 'lemx-S2'] (brightest: ['scox1', 'gx5-1', 'lemx-S1', 'lemx-S2'])

